# Bolted Field Splice Designer — guide & synthetic example

`civilpy.structural.aashto.lrfd.design_splice` designs a **bolted field splice
for a steel plate girder** to AASHTO LRFD 6.13.6.1: given the girder section on
each side of the splice, the splice-centerline loads, and the bolt / splice-plate
/ clearance selections, it

1. **sizes** the flange and web bolt groups,
2. **lays out** the bolt pattern (gage, pitch, edge, end) and splice-plate sizes,
3. **checks** the AASHTO limit states and reports OK / NOTICE per article.

This notebook has two parts:

* **Part 1 — Using the designer**: a quick-start example, a reference for every
  input, how to read the result object, a design-iteration example, and a
  bolt-layout drawing.
* **Part 2 - Synthetic example**: an invented rolled-beam load case and
  plate selections, with calculated checks displayed for inspection.

**AASHTO articles exercised:** 6.13.6.1.3b (flange design force *Pfy*),
6.13.6.1.3c (web forces / *Hw*), 6.13.2.7 (bolt shear), 6.13.2.8 (slip),
6.13.2.9 (bearing), 6.13.4 (block shear), 6.13.5.2 (net section), 6.13.6.1.4
(filler *R*), 6.13.2.6 (spacing), 6.10.9 (web shear), D6.1 (slab crushing).

**Units throughout: kip, inch, ksi**; moments are supplied in **kip-ft**.

The separate `nsba_splice_validation.ipynb` retains the public NSBA
plate-girder benchmark examples. This notebook uses no private project data.


In [ ]:
%matplotlib inline
import dataclasses

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

from civilpy.structural.aashto.lrfd import (
    design_splice, SpliceInput, Flange, GirderSide, SpliceLoads, BoltSpec,
    PlatePair, WebPlate,
)
from civilpy.structural.aashto.lrfd.bolted_field_splice import (
    STEEL_GRADES, BOLT_FU, STANDARD_HOLE_DIA,
)
from civilpy.structural.aashto.lrfd.steel import (
    SLIP_SURFACE_FACTORS, SLIP_HOLE_FACTORS, BOLT_PRETENSION,
)

pd.set_option("display.max_rows", 200)


def summary_table(d):
    # one-look summary of a SpliceDesign: bolts, layout, plate, verdict
    rows = []
    for c in d.components:
        rows.append({
            "component": c.name,
            "rows/side": c.bolt_rows,
            "bolts/side": c.total_bolts,
            "pitch": c.pitch,
            "plate w x L x t (in)":
                f"{c.plate_width} x {round(c.plate_length, 2)} x {c.plate_thickness}",
            "ok": c.ok,
        })
    return pd.DataFrame(rows)

# Part 1 — Using the designer

## 1.1 Quick start

A splice is described by one `SpliceInput`. Below is a symmetric composite
plate girder (same section each side). The seven structured inputs — `left`,
`right`, `loads`, `bolts`, `top_plates`, `bottom_plates`, `web_plate` — are
required; everything else (deck, bolt counts per row, spacing, clearances) has a
sensible default. `design_splice` returns a `SpliceDesign`.

In [ ]:
def quick_start_input():
    # one section, used on both sides of the splice
    side = GirderSide(
        top_flange=Flange("Grade 50", 1.0, 15.0),      # material, t (in), w (in)
        bottom_flange=Flange("Grade 50", 1.25, 17.0),
        web_material="Grade 50W", web_thickness=0.5, web_depth=60.0,
        haunch=1.0, stiffener_spacing_ft=15.0, stiffened=True,
    )
    return SpliceInput(
        left=side, right=side,
        loads=SpliceLoads(                              # unfactored, kip-ft / kip
            dc1_m=200, dc1_v=-60, dc2_m=40, dc2_v=-10, dw_m=40, dw_v=-10,
            ll_pos_m=1400, ll_pos_v=20, ll_neg_m=-900, ll_neg_v=-90,
            deck_cast_m=800, deck_cast_v=-55),
        bolts=BoltSpec(),                               # A325, 7/8 in, defaults
        top_plates=PlatePair("Grade 50", 0.75, 6.5, 0.6875, 15.0),
        bottom_plates=PlatePair("Grade 50", 0.875, 7.0, 0.8125, 17.0),
        web_plate=WebPlate("Grade 50", 0.375),
        deck_composite=True, deck_thickness=8.5, deck_eff_width=96.0, fc=4.0,
        flange_end=1.5, web_end=1.5,
    )


d_qs = design_splice(quick_start_input())
print("overall design ok:", d_qs.ok)
summary_table(d_qs)

## 1.2 Inputs reference

Every input and its default. The seven **structured** inputs are dataclasses;
the remaining **scalar** inputs live directly on `SpliceInput`.

In [ ]:
def _describe(cls, notes):
    rows = []
    for f in dataclasses.fields(cls):
        if f.default is not dataclasses.MISSING:
            default = f.default
        elif f.default_factory is not dataclasses.MISSING:      # noqa
            default = f.default_factory()
        else:
            default = "— (required)"
        rows.append({"field": f.name, "default": default,
                     "meaning / units": notes.get(f.name, "")})
    return pd.DataFrame(rows)


print("SpliceInput — scalar inputs (structured ones shown separately below):")
_describe(SpliceInput, {
    "left": "GirderSide on the left of the splice (see below)",
    "right": "GirderSide on the right of the splice",
    "loads": "SpliceLoads — unfactored splice-centerline demands",
    "bolts": "BoltSpec — bolt type / size / faying surface",
    "top_plates": "PlatePair — top-flange inner+outer splice plates",
    "bottom_plates": "PlatePair — bottom-flange splice plates",
    "web_plate": "WebPlate — web splice plate",
    "deck_composite": "deck acts compositely with the steel section",
    "deck_thickness": "structural deck thickness ts (in)",
    "deck_eff_width": "effective deck width for slab crushing (in)",
    "fc": "deck concrete strength f'c (ksi)",
    "top_flange_rows": "transverse bolt rows, top flange (per side)",
    "bottom_flange_rows": "transverse bolt rows, bottom flange (per side)",
    "web_rows": "vertical bolt lines, web (per side)",
    "bolt_spacing": "bolt pitch & gage within a group (in)",
    "flange_edge": "flange-plate transverse edge distance (in)",
    "flange_end": "flange-plate longitudinal end distance (in)",
    "web_edge": "web-plate edge distance (in)",
    "web_end": "web-plate end distance (in)",
    "web_weld_size": "web-to-flange weld size (in)",
    "web_weld_clearance": "clearance beside the web weld (in)",
    "girder_gap": "gap between girder ends at the splice (in)",
    "entering_tightening": "wrench entering & tightening clearance (in)",
    "design_year": "AASHTO edition; >=2017 uses the 0.56/0.45 bolt-shear "
                   "coefficient, earlier uses 0.48/0.38",
}).iloc[7:].reset_index(drop=True)

In [ ]:
tables = {
    "Flange (one flange, one side)": _describe(Flange, {
        "material": "steel grade (see grades table)",
        "thickness": "flange thickness (in)", "width": "flange width (in)"}),
    "GirderSide (one side of the splice)": _describe(GirderSide, {
        "top_flange": "Flange", "bottom_flange": "Flange",
        "web_material": "web steel grade", "web_thickness": "web t (in)",
        "web_depth": "clear web depth D (in)", "haunch": "haunch depth (in)",
        "stiffener_spacing_ft": "transverse stiffener spacing do (ft); None if "
                                "unstiffened",
        "stiffened": "web is transversely stiffened"}),
    "PlatePair (inner + outer flange splice plates)": _describe(PlatePair, {
        "material": "plate steel grade",
        "inner_thickness": "each inner plate t (in)",
        "inner_width": "each inner plate width (in)",
        "outer_thickness": "outer plate t (in)", "outer_width": "outer w (in)",
        "shear_planes": "bolt shear planes Ns (2 = double shear)"}),
    "WebPlate": _describe(WebPlate, {
        "material": "plate steel grade", "thickness": "each web plate t (in)",
        "shear_planes": "bolt shear planes Ns"}),
    "BoltSpec": _describe(BoltSpec, {
        "bolt_type": "A325 or A490", "diameter": "bolt diameter (in)",
        "flange_threads_excluded": "threads excluded from flange shear planes",
        "web_threads_excluded": "threads excluded from web shear planes",
        "surface_class": "faying-surface class Ks (A/B/C/D)",
        "hole_type": "standard / oversize / long_slot_perp / long_slot_par"}),
    "SpliceLoads (unfactored; _m = moment kip-ft, _v = shear kip)": _describe(
        SpliceLoads, {
            "dc1_m": "noncomposite dead load DC1", "dc1_v": "",
            "dc2_m": "composite dead load DC2", "dc2_v": "",
            "dw_m": "future wearing surface DW", "dw_v": "",
            "ll_pos_m": "positive live load + impact", "ll_pos_v": "",
            "ll_neg_m": "negative live load + impact", "ll_neg_v": "",
            "deck_cast_m": "deck-casting sequence", "deck_cast_v": ""}),
}
for title, tbl in tables.items():
    print(f"\n### {title}")
    display(tbl)

**Reference values** used internally — the allowed steel grades, bolt
ultimate strengths, standard hole diameters, faying-surface (Ks) and hole (Kh)
factors, and bolt pretension — pulled live from civilpy.

In [ ]:
print("Steel grades  (Fy, Fu ksi):")
display(pd.DataFrame([(g, fy, fu) for g, (fy, fu) in STEEL_GRADES.items()],
                     columns=["grade", "Fy", "Fu"]))
print("Bolt Fu (ksi):", BOLT_FU)
print("Standard hole diameter by bolt dia (in):", STANDARD_HOLE_DIA)
print("Surface class Ks (6.13.2.8):", SLIP_SURFACE_FACTORS)
print("Hole factor Kh (6.13.2.8):", SLIP_HOLE_FACTORS)
print("Min bolt pretension Pt (kip), Table 6.13.2.8-1:",
      {f"{g} {d}\"": pt for (g, d), pt in BOLT_PRETENSION.items()})

## 1.3 Reading the result

`design_splice` returns a `SpliceDesign`:

* `.factored_moments` / `.factored_shears` — the governing load combinations
  (Strength I ±, Service II ±, deck casting).
* `.components` — three `ComponentDesign`s: `top_flange`, `web`, `bottom_flange`
  (also on the design as attributes). Each carries the bolt count, layout,
  plate size, and a list of `.checks`.
* `.ok` — True only if every component's checks pass.

Each `.checks` entry is a `CheckResult` with `.article`, `.name`, `.capacity`,
`.demand`, `.factored_capacity` (= `phi * capacity`), `.ratio`
(`factored_capacity / demand`, ≥ 1 passes) and `.ok`.

In [ ]:
d = d_qs
print("type:", type(d).__name__, "| overall ok:", d.ok)
print("factored moments (kip-ft):",
      {k: round(v, 1) for k, v in d.factored_moments.items()})

# anatomy of one component
c = d.top_flange
print(f"\n{c.name}: {c.total_bolts} bolts in {c.bolt_rows} rows; design force "
      f"Pfy = {c.design_force:.1f} kip; filler R = {c.extra['filler_R']:.3f}")
pd.DataFrame([{
    "article": chk.article, "check": chk.name,
    "phi*capacity": round(chk.factored_capacity, 1),
    "demand": None if chk.demand is None else round(chk.demand, 1),
    "ratio": None if chk.ratio is None else round(chk.ratio, 2),
    "ok": chk.ok,
} for chk in c.checks])

**Interpreting the verdicts.** A check with a demand fails when its
capacity/demand ratio drops below 1.0. Failed serviceability, flange-capacity,
and strength checks are available through `ok is False`; inspect every
reported check when assessing the design:


In [ ]:
failing = [(c.name, chk.article, chk.name) for c in d.components
           for chk in c.checks if chk.ok is False]
print("failing checks:", failing or "none — all limit states pass")

# filler plates: engaged only where the two sides differ in flange thickness
print("\nfiller-plate reduction R per flange (1.0 = no filler):")
for c in (d.top_flange, d.bottom_flange):
    print(f"  {c.name:14} R = {c.extra['filler_R']:.4f}")

## 1.4 Design iteration

Because the plates and bolt selections are *inputs*, designing is a loop: adjust
a value, re-run, read the result. `SpliceInput` is a frozen dataclass, so
`dataclasses.replace` makes a tweaked copy. Here we swap 7/8-in bolts for 1-in
bolts and watch the bolt counts and layout respond.

In [ ]:
base = quick_start_input()
bigger = dataclasses.replace(base, bolts=BoltSpec(diameter=1.0))

cmp = summary_table(design_splice(base))[["component", "bolts/side", "pitch"]]
cmp = cmp.rename(columns={"bolts/side": "7/8in bolts", "pitch": "7/8in pitch"})
big = summary_table(design_splice(bigger))[["bolts/side", "pitch"]]
cmp["1in bolts"] = big["bolts/side"].values
cmp["1in pitch"] = big["pitch"].values
cmp

## 1.5 Bolt-layout drawing

A quick plan/elevation of the designed bolt pattern and splice plates — the
notebook's answer to the workbook *Figures* sheet. Coordinates are reconstructed
from the `ComponentDesign` layout fields (gage, pitch, edge, end), with the
splice centerline at 0.

In [ ]:
def _flange_xy(c):
    n_per_group = c.bolt_rows // 2
    ys, y = [], c.edge
    for g in range(2):
        for i in range(n_per_group):
            ys.append(y)
            if i < n_per_group - 1:
                y += c.gage_bolts
        if g == 0:
            y += c.gage_groups
    ys = [v - c.plate_width / 2 for v in ys]
    cols = c.total_bolts // c.bolt_rows
    xs = []
    for col in range(cols):
        d = c.pitch_groups / 2 + col * c.pitch
        xs += [d, -d]
    return xs, ys


def _web_xy(c):
    per_row = c.total_bolts // c.bolt_rows
    xs = []
    for line in range(c.bolt_rows):
        d = c.gage_groups / 2 + line * c.gage_bolts
        xs += [d, -d]
    ys = [(-(per_row - 1) / 2 + k) * c.pitch for k in range(per_row)]
    return xs, ys


def plot_splice_layout(design, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(13, 5))
    panels = [("top_flange", design.top_flange, "plan"),
              ("web", design.web, "elev"),
              ("bottom_flange", design.bottom_flange, "plan")]
    for ax, (name, c, kind) in zip(axes, panels):
        if kind == "plan":
            xs, ys = _flange_xy(c)
            L, W = c.plate_length, c.plate_width           # x = long, y = width
        else:
            xs, ys = _web_xy(c)
            L, W = c.plate_width, c.plate_length           # x = width, y = height
        ax.add_patch(mpatches.Rectangle((-L / 2, -W / 2), L, W, fill=False,
                                        edgecolor="0.5", linewidth=1.2))
        for x in xs:
            for y in ys:
                ax.plot(x, y, "o", ms=5, color="#1f4e79")
        ax.axvline(0, ls="--", lw=0.8, color="crimson")   # splice centerline
        ax.set_title(f"{name}\n{c.total_bolts} bolts/side, plate "
                     f"{c.plate_width}x{round(c.plate_length, 1)}x"
                     f"{c.plate_thickness}", fontsize=9)
        ax.set_aspect("equal")
        ax.margins(0.15)
        ax.set_xlabel("in")
    fig.suptitle(title or "Bolt layout (splice centerline dashed)")
    fig.tight_layout()
    plt.show()


plot_splice_layout(d_qs, "Quick-start splice — designed bolt layout")

# Part 2 - Synthetic rolled-beam example

The loads, deck dimensions, and plate selections below are invented for this
demonstration; they do not describe a project or reproduce a private workbook.
W-shape dimensions use the bundled AISC catalog. Read the calculated checks
below; this example is not an independent validation against a project design.


In [ ]:
from civilpy.structural.aashto.lrfd import design_rolled_splice

loads = SpliceLoads(
    dc1_m=20.0, dc1_v=-12.0, dc2_m=5.0, dc2_v=-3.0,
    dw_m=8.0, dw_v=-5.0, ll_pos_m=280.0, ll_neg_m=-160.0,
    ll_neg_v=-30.0,
)
plates = PlatePair("Grade 50", 0.5, 5.0, 0.5, 12.5, 2)
demo = design_rolled_splice(
    "W24X131", "W24X104", loads, deck_thickness=8.0,
    deck_eff_width=96.0, rebar_area=6.0,
    bolts=BoltSpec("A325", 0.875, flange_threads_excluded=False,
                   web_threads_excluded=False, surface_class="C",
                   hole_type="oversize"),
    top_plates=plates, bottom_plates=plates,
    web_plate=WebPlate("Grade 50", 0.5, 2),
    top_flange_rows=2, bottom_flange_rows=2, web_rows=4,
    bolt_spacing=3.0, flange_edge=1.5, flange_end=1.5,
    web_edge=1.5, web_end=1.5, design_year=2016,
)
display(summary_table(demo))
for component in demo.components:
    display(pd.DataFrame([vars(check) for check in component.checks]))
assert demo.ok
